# n-step TD Prediction -- Mapping the Pseudocode to the Equations

Sutton & Barto Section 7.1 ("n-step TD Prediction"). The equations (the $n$-step return and the update rule) are usually the easy part; the pseudocode box is where people get stuck, because it introduces a *second* time index ($\tau$) and a bookkeeping detail (storing things mod $n+1$) that don't appear in the equations at all. This notebook maps the two together explicitly, with a fully worked trace.

Reward-indexing note: this notebook uses Sutton & Barto's own convention ($R_{t+1}$ is the reward from taking $A_t$ in $S_t$), matching the pseudocode itself and `Random_Walk_MRP.ipynb`/`Windy_Gridworld.ipynb` in this folder.

## 1. The equations you already have

Sutton & Barto write $V_t:\mathcal S\to\mathbb R$ for "the estimate *at time $t$* of $v_\pi$" -- $V$ changes online as the episode runs, so a bare "$V$" is ambiguous about *which* snapshot of it is meant. Written precisely, with that time-subscript:

The $n$-step return (book Eq. 7.1):
$$G_{t:t+n} \doteq R_{t+1}+\gamma R_{t+2}+\dots+\gamma^{n-1}R_{t+n}+\gamma^n V_{t+n-1}(S_{t+n}), \qquad n\ge 1,\ 0\le t\le T-n$$
and if $t+n\ge T$ (not enough real steps remain to form a full $n$-step lookahead), $G_{t:t+n}\doteq G_t$ -- the ordinary complete (Monte Carlo) return, with no bootstrap term at all.

The update (book Eq. 7.2):
$$V_{t+n}(S_t) \doteq V_{t+n-1}(S_t) + \alpha\big[G_{t:t+n} - V_{t+n-1}(S_t)\big], \qquad 0\le t< T$$
(and $V_{t+n}(s)=V_{t+n-1}(s)$ for every other state $s\ne S_t$).

Why the subscript matters here specifically: "No real algorithm can use the $n$-step return until after it has seen $R_{t+n}$ and computed $V_{t+n-1}$" (the book's own words) -- $V_{t+n-1}(S_{t+n})$ means precisely *the value table as it stood right before this particular update*, not some fixed or final value. The subscripts $t+n$ (left side) and $t+n-1$ (right side) mark "immediately after this one update" vs. "immediately before it."

That's the whole algorithm, conceptually: for each state visited during an episode, form its $n$-step return (using the *current* $V$) and move $V$ a step toward it. The pseudocode's job is just to say, mechanically, *when* each of these updates can actually happen as the episode unfolds in real time -- and that's where the extra index ($\tau$) comes from.


## 2. The pseudocode box, reproduced

```
n-step TD for estimating V ≈ v_π

Input: a policy π
Algorithm parameter: step size α ∈ (0,1], a positive integer n
Initialize V(s) arbitrarily, for all s ∈ S
All store and access operations (for S_t and R_t) can take their index mod n+1

Loop for each episode:
    Initialize and store S_0 ≠ terminal
    T ← ∞
    Loop for t = 0, 1, 2, ...:
        If t < T, then:
            Take an action according to π(·|S_t)
            Observe and store the next reward as R_{t+1} and the next state as S_{t+1}
            If S_{t+1} is terminal, then: T ← t + 1
        τ ← t − n + 1    (τ is the time whose state's estimate is being updated)
        If τ ≥ 0:
            G ← Σ_{i=τ+1}^{min(τ+n, T)} γ^{i−τ−1} R_i
            If τ + n < T, then: G ← G + γ^n V(S_{τ+n})    (this G is G_{τ:τ+n})
            V(S_τ) ← V(S_τ) + α[G − V(S_τ)]
    Until τ = T − 1
```

## 3. The key insight: two clocks, not one

$t$ is the **simulation clock** -- it tracks how far the environment has actually been stepped forward (the index of the most recently observed transition). $\tau$ is a *derived*, **lagging** index -- it tracks *which state's value estimate is being updated this iteration*. They are related by
$$\tau = t-n+1 \quad\Longleftrightarrow\quad \tau+n = t+1$$

This relationship is not arbitrary -- it's forced by what $G_{\tau:\tau+n}$ needs. To form the $n$-step return for state $S_\tau$, you need the $n$ rewards $R_{\tau+1},\dots,R_{\tau+n}$ and (if not yet terminal) the bootstrap value $V(S_{\tau+n})$. Since $\tau+n=t+1$, the state and reward you need for *this* update ($S_{\tau+n}=S_{t+1}$, and $R_{\tau+n}=R_{t+1}$) are **exactly the ones the pseudocode just stored one line earlier, in this same iteration**. That's the whole design: every outer-loop iteration takes (at most) one real step forward in the environment, which *simultaneously* supplies the one new piece of information needed to finally resolve the update for the state $n-1$ steps behind it.

So "$\tau\leftarrow t-n+1$" isn't really introducing new math -- it's just naming "which state can be updated *right now*, given what was just observed." The $G$ line in the pseudocode, $G\leftarrow\sum_{i=\tau+1}^{\min(\tau+n,T)}\gamma^{i-\tau-1}R_i$ (plus the bootstrap term if $\tau+n<T$), is *literally* $G_{\tau:\tau+n}$ from section 1 -- just written with $\tau$ in place of $t$, because by the time we're forming it, $\tau$ (not the current simulation time $t$) is the state actually being updated.

*(The "mod $n+1$" storage note is a separate, purely implementation-level point: at any moment you only ever need $S_\tau,\dots,S_{\tau+n}$ -- $n+1$ consecutive states -- and $R_{\tau+1},\dots,R_{\tau+n}$ simultaneously in memory, so a circular buffer of size $n+1$ suffices instead of storing the whole episode. It carries no conceptual content beyond that.)*

*(Likewise, the pseudocode's "$G\leftarrow G+\gamma^n V(S_{\tau+n})$" writes a bare $V$, with no time-subscript, while section 1's Eq. 7.1 writes $V_{t+n-1}(S_{t+n})$ precisely. These are the same object, not a discrepancy: the running code has exactly one mutable array, and whatever is stored there *at the moment this line reads it* automatically equals $V_{t+n-1}$, since nothing else writes to it in between. The subscript only needs to be made explicit in the equations, where "$V$" alone would be ambiguous about which snapshot in time is meant.)*

## 4. Before any update is possible: $\tau<0$

For $t=0,1,\dots,n-2$: $\tau=t-n+1<0$, so the "If $\tau\ge 0$" check fails and **no update happens at all** for these first $n-1$ iterations. This is just the obvious fact that you cannot form an $n$-step return for $S_0$ until you've observed $n$ real rewards -- the algorithm spends its first $n-1$ iterations purely accumulating data, with nothing yet to update.

## 5. After termination: why the loop keeps going

Once a terminal transition is observed, $T\leftarrow t+1$ is set (previously $T=\infty$). From then on, at every later iteration, "If $t<T$" is false -- there is nothing left to observe, so **no more real environment steps are taken**. But the outer loop keeps incrementing $t$ anyway, purely to keep advancing $\tau=t-n+1$ forward: there are up to $n-1$ states near the end of the episode whose updates couldn't be resolved yet in real time (forming their full $n$-step lookahead would have required peeking past the terminal state, which doesn't exist). This is exactly why the loop's stopping condition is "**Until $\tau=T-1$**" rather than "until $t=T-1$" -- a handful of extra, step-free iterations are needed purely to flush out these last few delayed updates.

This is also what the summation bound $\min(\tau+n,T)$ and the bootstrap condition "if $\tau+n<T$" are for: for a state $\tau$ close to the end, $\tau+n$ can exceed $T$ (there is no $S_{\tau+n}$ beyond the terminal state). In that case the reward sum simply stops at $R_T$ instead of $R_{\tau+n}$, and the bootstrap term is skipped entirely -- which is *exactly* the "$t+n\ge T\Rightarrow G_{t:t+n}=G_t$" branch from section 1, just automatically enforced by the index arithmetic rather than stated as a separate case.

## 6. Fully worked trace ($n=3$, episode length $T=5$)

Concrete trajectory: $S_0,S_1,\dots,S_5{=}\text{terminal}$, with rewards $R_1,\dots,R_5$ (5 real transitions). The outer loop runs $t=0,1,\dots,6$ (it must reach $\tau=T-1=4$, i.e. $t=4+n-1=6$, before stopping).

| $t$ | real step taken? ($t<T$) | $S_{t+1}$ terminal? | $\tau=t{-}2$ | update? | $G$ after the sum line only | bootstrap line adds... | **final $G$ used in the update** | $V(S_\tau)$ updated |
|---|---|---|---|---|---|---|---|---|
| 0 | yes, observe $R_1,S_1$ | no | $-2$ | no | -- | -- | -- | -- |
| 1 | yes, observe $R_2,S_2$ | no | $-1$ | no | -- | -- | -- | -- |
| 2 | yes, observe $R_3,S_3$ | no | $0$ | **yes** | $R_1+\gamma R_2+\gamma^2 R_3$ | yes ($\tau{+}n{=}3<5$): $+\gamma^3V(S_3)$ | $R_1+\gamma R_2+\gamma^2 R_3+\gamma^3V(S_3)$ | $V(S_0)$ |
| 3 | yes, observe $R_4,S_4$ | no | $1$ | **yes** | $R_2+\gamma R_3+\gamma^2 R_4$ | yes ($4<5$): $+\gamma^3V(S_4)$ | $R_2+\gamma R_3+\gamma^2 R_4+\gamma^3V(S_4)$ | $V(S_1)$ |
| 4 | yes, observe $R_5,S_5$ | **yes** $\Rightarrow T\leftarrow 5$ | $2$ | **yes** | $R_3+\gamma R_4+\gamma^2 R_5$ | no ($\tau{+}n{=}5\not<5$) -- sum already reaches $R_T$ | $R_3+\gamma R_4+\gamma^2 R_5$ (unchanged -- no bootstrap line fires) | $V(S_2)$ |
| 5 | no ($t\ge T$) | -- | $3$ | **yes** | $R_4+\gamma R_5$ (only 2 terms -- $\min(\tau{+}n,T)=5$) | no ($6\not<5$) | $R_4+\gamma R_5$ (unchanged) | $V(S_3)$ |
| 6 | no ($t\ge T$) | -- | $4$ | **yes** | $R_5$ (1 term) | no ($7\not<5$) | $R_5$ (unchanged) | $V(S_4)$ |

(The middle two columns mirror the pseudocode's two separate lines for $G$ -- the sum, then the conditional bootstrap addition -- so you can see which piece comes from which line; the **final $G$** column is the single value actually plugged into $V(S_\tau)\leftarrow V(S_\tau)+\alpha[G-V(S_\tau)]$, i.e. the sum-line result *plus* the bootstrap term whenever the bootstrap line fires, or just the sum-line result unchanged when it doesn't.)

At $t=6$, $\tau=4=T-1$, so the loop stops. Every non-terminal state $S_0,\dots,S_4$ got updated exactly once, each with its correct $n$-step (or correctly-shortened, near the end) return -- $S_0$ and $S_1$ got the full bootstrapped 3-step return; $S_2$'s 3-step return happened to land exactly on the terminal reward (boundary case, no bootstrap needed); $S_3$ and $S_4$, being within $n-1$ steps of the end, automatically got shortened, fully Monte-Carlo-style returns (2 and 1 real reward terms respectively) -- exactly matching the $t+n\ge T$ branch from section 1, with no special-casing required in the code.

## 7. Summary: pseudocode <-> equation dictionary

| Pseudocode element | What it means in the equations |
|---|---|
| $t$ | The simulation clock -- index of the most recently observed real transition. Advances by exactly 1 every outer-loop iteration (while $t<T$). |
| $\tau \leftarrow t-n+1$ | The state currently eligible for an update. Defined so that $\tau+n=t+1$: the data this update needs ($S_{\tau+n}, R_{\tau+n}$) is exactly what was just observed this same iteration. |
| "If $\tau\ge 0$" | Guards against trying to update before $n$ real rewards have been observed (section 4). |
| $G\leftarrow\sum_{i=\tau+1}^{\min(\tau+n,T)}\gamma^{i-\tau-1}R_i$ | $G_{\tau:\tau+n}$'s reward sum from section 1, with $\tau$ (not $t$) as the state being targeted; the $\min(\cdot,T)$ automatically shortens the sum near the end of the episode. |
| "If $\tau+n<T$, $G\leftarrow G+\gamma^nV(S_{\tau+n})$" | The bootstrap term from section 1's $G_{t:t+n}$ formula -- included only when $S_{\tau+n}$ exists (isn't past the terminal state). |
| $V(S_\tau)\leftarrow V(S_\tau)+\alpha[G-V(S_\tau)]$ | Exactly the update rule from section 1, $V_{t+n}(S_t)\doteq V_{t+n-1}(S_t)+\alpha[G_{t:t+n}-V_{t+n-1}(S_t)]$ (**Eq. 7.2**), with $\tau$ playing the role of $t$. The pseudocode's bare $V$ is $V_{t+n-1}$/$V_{t+n}$ before/after this line, respectively (section 3). |
| "Until $\tau=T-1$" | Keep going until every non-terminal state $S_0,\dots,S_{T-1}$ has been updated once -- including the post-termination iterations needed to flush out the last $n-1$ delayed updates (section 5). |
| "indices mod $n+1$" | Pure memory optimization: a circular buffer of size $n+1$ holds everything needed ($S_\tau,\dots,S_{\tau+n}$) without storing the whole episode. No conceptual content. |

## 8. Applying this to the 19-state Random Walk (Example 7.1)

Same algorithm as sections 1-7, now on a concrete environment: 19 non-terminal states in a line, equiprobable left/right random policy, reward $-1$ on reaching the left terminal, $+1$ on reaching the right terminal, $0$ otherwise.

**Encoding** (matching `Random_Walk_MRP.ipynb`'s convention): states `0..20`, where `0` = left terminal, `20` = right terminal, `1..19` = the actual non-terminal states, `START_STATE = 10` (the center).

**True values, derived analytically.** Under the equiprobable random policy, the Bellman equation for non-terminal state $s\in\{1,\dots,19\}$ is
$$v(s) = 0.5\,\big[r(s,\text{left}) + v(s-1)\big] + 0.5\,\big[r(s,\text{right}) + v(s+1)\big]$$
Reward is $0$ everywhere except on the two edge transitions, so for the interior states ($s=2,\dots,18$) this is just
$$v(s) = 0.5\,v(s-1) + 0.5\,v(s+1)$$
The two boundary states need their actual reward plugged in:
$$v(1) = 0.5\,[-1 + 0] + 0.5\,v(2) = 0.5(-1) + 0.5\,v(2), \qquad v(19) = 0.5\,v(18) + 0.5\,[1 + 0] = 0.5\,v(18) + 0.5(1)$$
These boundary equations match the *interior* recursion exactly if we define virtual boundary values $v(0):=-1$ and $v(20):=+1$ (i.e. the reward received on entering a terminal state stands in for "the value just across the edge"). So all 19 equations collapse to one recursion, $v(s)=0.5\,v(s-1)+0.5\,v(s+1)$ for $s=1,\dots,19$, with $v(0)=-1,\ v(20)=1$ as boundary conditions.

Try the linear ansatz $v(s) = as+b$. Substituting: $as+b = 0.5\,[a(s-1)+b] + 0.5\,[a(s+1)+b] = as+b$ -- holds identically for *any* $a,b$ (expected: one recursion, two unknowns, so the boundary conditions are what pin it down, not the recursion itself). Applying the boundary conditions:
$$v(0)=-1 \;\Rightarrow\; b=-1, \qquad v(20)=1 \;\Rightarrow\; 20a+b=1 \;\Rightarrow\; 20a-1=1 \;\Rightarrow\; a=\tfrac{1}{10}$$
So:
$$v(s) = \frac{s}{10} - 1 = \frac{s-10}{10}, \qquad s=1,\dots,19$$
A linear ramp from $v(1)=-0.9$ to $v(19)=0.9$, zero at the center ($v(10)=0$) -- matching Figure 7.2's description of true values ranging from $-1$ to $1$.

In [1]:
import numpy as np

START_STATE = 10
LEFT_TERMINAL = 0
RIGHT_TERMINAL = 20
TERMINAL_STATES = (LEFT_TERMINAL, RIGHT_TERMINAL)
NON_TERMINAL_STATES = list(range(1, 20))  # 1..19

TRUE_VALUES = np.zeros(21)
TRUE_VALUES[NON_TERMINAL_STATES] = [(s - 10) / 10 for s in NON_TERMINAL_STATES]
TRUE_VALUES[LEFT_TERMINAL] = 0.0   # terminal states themselves still have value 0 --
TRUE_VALUES[RIGHT_TERMINAL] = 0.0  # the +-1 is a *reward*, received on entering them, not their own value

TRUE_VALUES[NON_TERMINAL_STATES]

array([-0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1,  0. ,  0.1,
        0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9])

### A shared `step` helper

Both `generate_episode` (below) and `n_step_td_evaluate_episode` (further down) need to take one random left/right move and compute its reward -- rather than duplicate that logic in both places, factor it out once.

In [9]:
def step(state):
    """Take one random left/right move from `state`. Returns (next_state, reward)."""
    move = -1 if np.random.rand() < 0.5 else 1
    next_state = state + move
    if next_state == LEFT_TERMINAL:
        reward = -1.0
    elif next_state == RIGHT_TERMINAL:
        reward = 1.0
    else:
        reward = 0.0
    return next_state, reward

In [11]:
# Quick sanity check: run step() many times from a few states and confirm
# only the two expected (next_state, reward) outcomes ever occur
np.random.seed(0)
assert {step(1) for _ in range(200)} == {(0, -1.0), (2, 0.0)}

np.random.seed(0)
assert {step(19) for _ in range(200)} == {(18, 0.0), (20, 1.0)}

np.random.seed(0)
assert {step(10) for _ in range(200)} == {(9, 0.0), (11, 0.0)}

print("step() gives the right next_state/reward pairs from edge and middle states")

step() gives the right next_state/reward pairs from edge and middle states


### Episode generation

Plays out one episode under the equiprobable random policy, starting from `START_STATE`, until a terminal state is reached. Reward is $-1$ on the transition into the left terminal, $+1$ on the transition into the right terminal, $0$ otherwise -- unlike the 5-state walk, *both* ends now carry a nonzero reward.

In [12]:
def generate_episode(start_state=START_STATE):
    state = start_state
    states = [state]
    rewards = []
    while state not in TERMINAL_STATES:
        next_state, reward = step(state)
        states.append(next_state)
        rewards.append(reward)
        state = next_state
    return states, rewards

In [13]:
# Quick sanity check
np.random.seed(0)
states, rewards = generate_episode()
print("states: ", states)
print("rewards:", rewards)

assert states[0] == START_STATE
assert states[-1] in TERMINAL_STATES
assert len(rewards) == len(states) - 1
# every intermediate reward is 0; the only nonzero reward is the last one,
# and its sign must match which terminal state was reached
assert all(r == 0.0 for r in rewards[:-1])
if states[-1] == LEFT_TERMINAL:
    assert rewards[-1] == -1.0
else:
    assert rewards[-1] == 1.0
print("generate_episode looks correct")

states:  [10, 11, 12, 13, 14, 13, 14, 13, 14, 15, 14, 15, 16, 17, 18, 17, 16, 15, 16, 17, 18, 19, 20]
rewards: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
generate_episode looks correct


### The $n$-step TD update

This implementation follows the pseudocode box in section 2 line-by-line, with **both** time indices explicit: `t` is the simulation clock (advances by taking a real step, same as the "If t < T" block), and `tau = t - n + 1` is the lagging index of the state actually being updated this iteration (section 3) -- rather than generating the whole episode first and only ever looping over `tau` (the earlier, offline-restructured version).

One simplification versus the book's literal pseudocode: `rewards` is stored **1-indexed** (`rewards[0]` is an unused placeholder), so `rewards[i]` *is* $R_i$ directly -- no `rewards[i-1]` translation needed, unlike the offline version in the cross-check cell below (`_n_step_td_evaluate_episode_offline`). `states[k]` already equals $S_k$ with no offset, as always. We also don't bother with the "indices mod $n+1$" circular buffer (section 3's closing note: pure memory optimization, no conceptual content) -- just plain growing Python lists, to keep the code focused on the `t`/`tau` logic the pseudocode is actually about. The single-step logic itself (sample a move, compute its reward) is factored into the shared `step()` helper above, rather than duplicated here and in `generate_episode`. The function returns this same `rewards` list unchanged (placeholder included) -- so `rewards[i] = R_i` holds both inside the function *and* in whatever it returns, with no shift between the two (an earlier version stripped the placeholder on return, which quietly introduced a second, different indexing convention just for the returned value -- removed, since nothing in this notebook uses that return anyway).

In [14]:
def n_step_td_evaluate_episode(values, n, alpha=0.1, gamma=1.0):
    """Run one episode, applying n-step TD updates as soon as each becomes
    possible -- a direct translation of the section-2 pseudocode, with t (the
    simulation clock) and tau = t - n + 1 (the state being updated) both
    explicit, exactly as in the book."""
    state = START_STATE
    states = [state]   # states[k] = S_k
    rewards = [None]   # rewards[k] = R_k for k >= 1; rewards[0] is an unused placeholder
    T = float("inf")
    t = 0

    while True:
        if t < T:
            next_state, reward = step(state)
            states.append(next_state)
            rewards.append(reward)
            if next_state in TERMINAL_STATES:
                T = t + 1
            state = next_state

        tau = t - n + 1   # tau is the time whose state's estimate is being updated
        if tau >= 0:
            upper = min(tau + n, T)
            G = 0.0
            for i in range(tau + 1, upper + 1):
                G += gamma ** (i - tau - 1) * rewards[i]
            if tau + n < T:
                G += gamma ** n * values[states[tau + n]]
            s_tau = states[tau]
            values[s_tau] += alpha * (G - values[s_tau])

        if tau == T - 1:
            break
        t += 1

    return states, rewards  # one convention throughout: states[k] = S_k, rewards[0] = None (placeholder,
                            # unused), rewards[i] = R_i for i >= 1

In [15]:
# Cross-check: this (t, tau) version must behave *identically* to the earlier
# offline version (same seed -> same values), since it's a refactor, not a
# change in algorithm. Re-defining the old version here just for comparison.
def _n_step_td_evaluate_episode_offline(values, n, alpha=0.1, gamma=1.0):
    states, rewards = generate_episode()
    T = len(rewards)
    for tau in range(T):
        upper = min(tau + n, T)
        G = 0.0
        for i in range(tau + 1, upper + 1):
            G += gamma ** (i - tau - 1) * rewards[i - 1]
        if tau + n < T:
            G += gamma ** n * values[states[tau + n]]
        s_tau = states[tau]
        values[s_tau] += alpha * (G - values[s_tau])
    return states, rewards

np.random.seed(42)
values_online = np.zeros(21)
for _ in range(500):
    n_step_td_evaluate_episode(values_online, n=4, alpha=0.1)

np.random.seed(42)
values_offline = np.zeros(21)
for _ in range(500):
    _n_step_td_evaluate_episode_offline(values_offline, n=4, alpha=0.1)

assert np.allclose(values_online, values_offline)
print("(t, tau) version matches the old offline version exactly -- "
      f"max abs diff: {np.max(np.abs(values_online - values_offline))}")

(t, tau) version matches the old offline version exactly -- max abs diff: 0.0


In [16]:
# Sanity check 1: n=1 must reduce exactly to ordinary TD(0)
def td0_evaluate_episode(values, alpha=0.1, gamma=1.0):
    states, rewards = generate_episode()
    for t in range(len(rewards)):
        s, s_next = states[t], states[t + 1]
        values[s] += alpha * (rewards[t] + gamma * values[s_next] - values[s])
    return states, rewards

np.random.seed(0)
values_n1 = np.zeros(21)
for _ in range(50):
    n_step_td_evaluate_episode(values_n1, n=1, alpha=0.1)

np.random.seed(0)
values_td0 = np.zeros(21)
for _ in range(50):
    td0_evaluate_episode(values_td0, alpha=0.1)

assert np.allclose(values_n1, values_td0), "n=1 should match ordinary TD(0) exactly"
print("n=1 matches TD(0) exactly -- max abs diff:", np.max(np.abs(values_n1 - values_td0)))

n=1 matches TD(0) exactly -- max abs diff: 0.0


In [17]:
# Sanity check 2: very large n must reduce to full Monte Carlo (no bootstrapping at all,
# since tau+n < T never holds once n exceeds the longest possible episode length)
def mc_evaluate_episode(values, alpha=0.1, gamma=1.0):
    states, rewards = generate_episode()
    T = len(rewards)
    G = 0.0
    returns = [0.0] * T
    for t in reversed(range(T)):
        G = rewards[t] + gamma * G
        returns[t] = G
    for t in range(T):
        s = states[t]
        values[s] += alpha * (returns[t] - values[s])
    return states, rewards

np.random.seed(0)
values_big_n = np.zeros(21)
for _ in range(50):
    n_step_td_evaluate_episode(values_big_n, n=10_000, alpha=0.1)

np.random.seed(0)
values_mc = np.zeros(21)
for _ in range(50):
    mc_evaluate_episode(values_mc, alpha=0.1)

assert np.allclose(values_big_n, values_mc), "very large n should match full Monte Carlo"
print("n=10,000 matches Monte Carlo exactly -- max abs diff:", np.max(np.abs(values_big_n - values_mc)))

n=10,000 matches Monte Carlo exactly -- max abs diff: 0.0


In [19]:
# Sanity check 3: for an intermediate n, repeated training should still move
# substantially toward the true values (not just match some other special-case
# method). Note: alpha=0.1 is a *constant*, non-decaying step size, so this
# never converges to the exact true values -- it settles into a steady-state
# fluctuation band around them (same constant-alpha noise floor documented in
# Random_Walk_MRP.ipynb and Short_Corridor.ipynb). Checked across 30 seeds,
# that band for n=4, alpha=0.1, 2000 episodes is RMS ~0.04-0.22 -- so the
# threshold below is deliberately generous, not a near-exact-convergence bar.
np.random.seed(0)
values_n4 = np.zeros(21)
for _ in range(2000):
    n_step_td_evaluate_episode(values_n4, n=4, alpha=0.1)

initial_error = np.sqrt(np.mean(TRUE_VALUES[NON_TERMINAL_STATES] ** 2))  # error of V_0 = 0 everywhere
final_error = np.sqrt(np.mean((values_n4[NON_TERMINAL_STATES] - TRUE_VALUES[NON_TERMINAL_STATES]) ** 2))
print(f"RMS error, initial (V=0): {initial_error:.4f}")
print(f"RMS error, after 2000 episodes (n=4): {final_error:.4f}")
assert final_error < initial_error / 2  # substantial, not exact, convergence
assert final_error < 0.3                # generous absolute bound, robust across seeds
print("n=4 moves substantially toward the true values, as expected")

RMS error, initial (V=0): 0.5477
RMS error, after 2000 episodes (n=4): 0.1422
n=4 moves substantially toward the true values, as expected


## 9. A more efficient $G$ computation (suffix discounted sums)

`n_step_td_evaluate_episode` above mirrors the pseudocode box literally, including its own inefficiency: recomputing $\sum_{i=\tau+1}^{\min(\tau+n,T)}\gamma^{i-\tau-1}R_i$ from scratch at every $\tau$ is $O(n)$ per update, even though consecutive windows ($\tau$ and $\tau+1$) share $n-1$ of their $n$ terms. Over a full episode that's $O(nT)$ total, when it could be $O(T)$.

**The standard fix** -- the same backward-recursive computation of the actual **return** $G_t$ already used elsewhere in this project (`mc_evaluate_episode` above, and REINFORCE in `Short_Corridor.ipynb`), stored once in an array rather than recomputed from scratch at every $\tau$:
$$G_t \doteq R_{t+1}+\gamma R_{t+2}+\dots+\gamma^{T-1-t}R_T, \qquad G_t = R_{t+1} + \gamma\, G_{t+1}, \quad G_T \doteq 0$$
This is Sutton & Barto's own $G_t$, nothing new -- the code below stores it in a plain Python list `G`, with `G[t]` $=G_t$. (The $n$-step return *target* actually plugged into the update -- what the pseudocode itself calls `G` -- is named `target` in this function instead, specifically to avoid a name clash with this array; see the code.)

One $O(T)$ backward pass builds every $G_t$. Then, because the tail terms telescope away exactly, any $n$-step reward-sum falls out of a single subtraction:
$$\sum_{k=0}^{n-1}\gamma^k R_{\tau+k+1} = G_\tau - \gamma^{n}\,G_{\tau+n}$$

**Why this telescopes, written out term by term.** Take $m=\tau+n$ (assume $m\le T$ for now) and write both full returns out completely:
$$G_\tau = \underbrace{R_{\tau+1}+\gamma R_{\tau+2}+\dots+\gamma^{n-1}R_{\tau+n}}_{\text{the }n\text{ terms we want}} + \underbrace{\gamma^n R_{\tau+n+1}+\gamma^{n+1}R_{\tau+n+2}+\dots+\gamma^{T-\tau-1}R_T}_{\text{everything after }\tau+n}$$
$$G_{\tau+n} = R_{\tau+n+1}+\gamma R_{\tau+n+2}+\dots+\gamma^{T-\tau-n-1}R_T$$
Multiplying the second line by $\gamma^n$:
$$\gamma^n\,G_{\tau+n} = \gamma^n R_{\tau+n+1}+\gamma^{n+1}R_{\tau+n+2}+\dots+\gamma^{T-\tau-1}R_T$$
which is *exactly* the second (underbraced) chunk of $G_\tau$ above -- same terms, same powers of $\gamma$, lined up exactly. Subtracting cancels it completely:
$$G_\tau-\gamma^n G_{\tau+n} = \underbrace{R_{\tau+1}+\gamma R_{\tau+2}+\dots+\gamma^{n-1}R_{\tau+n}}_{\text{the }n\text{ terms we want}} + 0 = \sum_{k=0}^{n-1}\gamma^k R_{\tau+k+1}$$
Intuitively: $G_\tau$ is "everything from $\tau+1$ to the end," $G_{\tau+n}$ is "everything from $\tau+n+1$ to the end" -- their difference is just "everything from $\tau+1$ up to $\tau+n$," i.e. exactly the $n$-term window. The $\gamma^n$ factor only exists to line up the discount powers before subtracting (since $G_{\tau+n}$'s own terms start at $\gamma^0$, but need to start at $\gamma^n$ from $\tau$'s point of view).

**Why $m=\min(\tau+n,T)$ in general, not just $\tau+n$.** The telescoping argument just above secretly assumed $\tau+n\le T$ -- it wrote out $G_\tau$ and $G_{\tau+n}$ as if both are ordinary returns starting from a real, in-episode state, which requires at least $n$ more real reward-steps to exist after $\tau$.

That assumption can fail near the end of an episode: if $\tau$ is within $n-1$ steps of termination, $\tau+n$ overshoots past $T$ -- there's no state $n$ steps ahead, because the episode already ended, so "$G_{\tau+n}$" doesn't refer to anything.

The fix is to use $m=\min(\tau+n,T)$ -- capped at the episode's actual length -- together with the convention $G_T\doteq 0$ (no reward exists after termination). The *same* formula, $G_\tau-\gamma^{m-\tau}G_m$, then handles both regimes automatically, with no extra branching in the math itself:
- **Normal case** ($\tau+n< T$, strict): $m=\tau+n$, giving exactly the $n$-term window derived above, *and* the bootstrap term is added (the pseudocode's own condition is "if $\tau+n<T$").
- **Near the end** ($\tau+n\ge T$): $m=T$, so $G_m=G_T=0$, and the formula collapses to $G_\tau-\gamma^{T-\tau}\cdot 0=G_\tau$ -- the *entire* remaining return, no truncation, no bootstrap.

(The equality case $\tau+n=T$ belongs with "near the end," not "normal," even though $m=\tau+n$ and $m=T$ are numerically the same number there. What actually decides which regime this is isn't the value of $m$ -- it's the bootstrap condition, and that condition is a *strict* inequality everywhere else in this notebook: the pseudocode's "if $\tau+n<T$" and section 1's "if $t+n\ge T$, $G_{t:t+n}\doteq G_t$" both put $\tau+n=T$ on the no-bootstrap side. So at that exact boundary, no bootstrap happens, and classifying it as "normal" would be numerically harmless but conceptually inconsistent with how the split is drawn everywhere else.)

That second case is exactly the "$t+n\ge T\Rightarrow G_{t:t+n}=G_t$" rule from section 1 (not enough real steps left for a true $n$-step lookahead, so fall back to the ordinary complete return) -- falling out for free from capping $m$, rather than needing to be handled as a separate special case. This is also exactly why the code computes `upper = min(tau + n, T)` rather than always using `tau + n` directly.

This uses only multiplication and powers, never division -- no numerical-stability question to even raise, unlike the earlier (division-based) attempt.

**Structural tradeoff:** this needs the *whole* episode's rewards before the backward pass can run, so it goes back to the *offline* structure (`generate_episode()` once, then two passes: backward to build $G$, forward to apply the updates) rather than the online $t$/$\tau$ interleaving. That's fine for prediction under a fixed policy -- nothing here needs the online structure, which (as section 3 already notes) is really only load-bearing for *control*, where behavior depends on the still-evolving value estimates.

The bootstrap term ($\gamma^n V(S_{\tau+n})$, when $\tau+n<T$) was already $O(1)$ -- a single array lookup plus one power -- so it's unchanged.

In [20]:
def n_step_td_evaluate_episode_fast(values, n, alpha=0.1, gamma=1.0):
    """Same algorithm as n_step_td_evaluate_episode, but the n-step
    reward-sum is computed from a precomputed array of actual returns
    G[t] = G_t (one backward O(T) pass) instead of a fresh O(n) sum at
    every tau. Offline, not online -- see markdown above for why that's
    fine here."""
    states, rewards = generate_episode()  # rewards[k] = R_{k+1}, matching generate_episode's own convention
    T = len(rewards)

    G = [0.0] * (T + 1)  # G[t] = the actual return G_t; G[T] = 0 by convention (no reward after termination)
    for t in reversed(range(T)):
        G[t] = rewards[t] + gamma * G[t + 1]

    for tau in range(T):
        upper = min(tau + n, T)
        reward_sum = G[tau] - (gamma ** (upper - tau)) * G[upper]
        target = reward_sum
        if tau + n < T:
            target += (gamma ** n) * values[states[tau + n]]
        s_tau = states[tau]
        values[s_tau] += alpha * (target - values[s_tau])

    return states, rewards

In [21]:
# Cross-check against n_step_td_evaluate_episode: same seed, same values, for
# several n and several gamma across many seeds
mismatches = 0
worst_diff = 0.0
for gamma in [1.0, 0.99, 0.95, 0.9, 0.7, 0.5]:
    for n in [1, 2, 3, 5, 10, 25, 10_000]:
        for seed in range(50):
            np.random.seed(seed)
            v_slow = np.zeros(21)
            for _ in range(30):
                n_step_td_evaluate_episode(v_slow, n=n, alpha=0.1, gamma=gamma)
            np.random.seed(seed)
            v_fast = np.zeros(21)
            for _ in range(30):
                n_step_td_evaluate_episode_fast(v_fast, n=n, alpha=0.1, gamma=gamma)
            diff = np.max(np.abs(v_slow - v_fast))
            worst_diff = max(worst_diff, diff)
            if diff > 1e-9:
                mismatches += 1
                print(f"mismatch at gamma={gamma}, n={n}, seed={seed}, diff={diff:.3e}")

assert mismatches == 0
print(f"n_step_td_evaluate_episode_fast matches the original across 6 gamma values x "
      f"7 n values x 50 seeds (2,100 combinations) -- worst max abs diff: {worst_diff:.2e} "
      f"(floating-point noise, not a real discrepancy).")

n_step_td_evaluate_episode_fast matches the original across 6 gamma values x 7 n values x 50 seeds (2,100 combinations) -- worst max abs diff: 3.89e-16 (floating-point noise, not a real discrepancy).


## 10. A third variant: the sliding window (division-based, online)

Like section 9, this starts from the same complaint about `n_step_td_evaluate_episode`: recomputing $\sum_{i=\tau+1}^{\min(\tau+n,T)}\gamma^{i-\tau-1}R_i$ from scratch at every $\tau$ is $O(n)$ per update, $O(nT)$ per episode, even though consecutive windows ($\tau$ and $\tau+1$) share $n-1$ of their $n$ terms. The idea here is the most direct fix imaginable: don't recompute the sum, just *slide* it -- drop the term that leaves, add the term that enters. A third way of actually doing that, kept explicitly alongside the other two for comparison rather than replacing either. Unlike section 9's suffix-sum approach, this one *does* need division -- and that's not an oversight, it's forced by what a $\tau$-relative sliding window actually requires.

**The picture, before any symbols.** For $n=3$, here's the window at $\tau=0,1,2$, with the power of $\gamma$ each reward is weighted by directly underneath it:

```
rewards:        R_1    R_2    R_3    R_4    R_5    R_6   ...

tau=0 window:  [R_1    R_2    R_3]
     weight:    g^0    g^1    g^2

tau=1 window:         [R_2    R_3    R_4]
     weight:           g^0    g^1    g^2

tau=2 window:                [R_3    R_4    R_5]
     weight:                  g^0    g^1    g^2
```

Follow $R_3$ across the three rows: it's weighted $\gamma^2$ when $\tau=0$ (it's the farthest-out term in that window), $\gamma^1$ when $\tau=1$, and $\gamma^0$ when $\tau=2$ (now it's the nearest term). The reward itself never changes -- only its weight does, because the weight is always measured as "distance from the window's own left edge," and that left edge keeps moving. Sliding the window by one position does two things to every reward that survives inside it: its weight drops by one power of $\gamma$ (what was $\gamma^1$ becomes $\gamma^0$, what was $\gamma^2$ becomes $\gamma^1$, etc.) -- and separately, the term now outside on the left ($R_{\tau+1}$) is dropped, while a new term enters on the right, at the farthest/faintest weight $\gamma^{n-1}$.

That "every surviving weight drops by one power of $\gamma$" step is exactly the rescaling that forces a division -- lowering every exponent by 1 at once, for a whole window of terms in a single scalar sum, means dividing the sum by $\gamma$. The rest of this section just makes that picture precise.

**Why division is unavoidable here, specifically.** The window's weights are $\gamma^0,\dots,\gamma^{n-1}$ *relative to $\tau$*. When $\tau$ advances to $\tau+1$, every surviving term's exponent must decrease by 1 (one step closer to the new reference point) -- and decreasing an exponent by 1 means dividing by $\gamma$. There's no way to "age" a geometrically-weighted window using only addition and multiplication if the result has to stay $\tau$-relative. (One *could* keep the window in a globally-referenced frame instead, using only addition/multiplication for the window update itself -- but converting back to the $\tau$-relative value actually needed for the TD update, since $V$ lives in ordinary unscaled value-space, still needs one division; it doesn't eliminate the operation, just relocates it.) This is the real reason section 9's suffix-sum approach has no division to raise a numerical-stability question about in the first place: $G_\tau$ and $G_m$ are each an independently well-conditioned backward-recursive sum, with no reference-frame shifting involved at all -- not a cleverer version of the same trick, a genuinely different mechanism.

**Deriving the recursion, term by term.** Write $m_\tau=\min(\tau+n,T)$, so $\text{reward\_sum}(\tau)=\sum_{i=\tau+1}^{m_\tau}\gamma^{i-\tau-1}R_i$. Peel off the first term:
$$\text{reward\_sum}(\tau) = R_{\tau+1} + \gamma\underbrace{\sum_{i=\tau+2}^{m_\tau}\gamma^{i-\tau-2}R_i}_{=:S}$$
so $S = \dfrac{\text{reward\_sum}(\tau)-R_{\tau+1}}{\gamma}$ -- this is where the division comes from: isolating $S$ means undoing the $\gamma$ that was factored out.

Now compare $S$ to $\text{reward\_sum}(\tau+1)=\sum_{i=\tau+2}^{m_{\tau+1}}\gamma^{i-\tau-2}R_i$ -- *same* summand, *same* starting index $i=\tau+2$, differing only in the upper bound ($m_\tau$ vs. $m_{\tau+1}$):
- **If $\tau+n+1\le T$:** then $\tau+n\le T$ too, so $m_\tau=\tau+n$ and $m_{\tau+1}=\tau+n+1=m_\tau+1$ -- exactly one new term beyond $S$, at $i=\tau+n+1$, weighted $\gamma^{(\tau+n+1)-\tau-2}=\gamma^{n-1}$. So $\text{reward\_sum}(\tau+1) = S+\gamma^{n-1}R_{\tau+n+1}$.
- **If $\tau+n+1> T$:** then $\tau+n\ge T$, so both $m_\tau$ and $m_{\tau+1}$ are already clipped at $T$ -- $m_\tau=m_{\tau+1}=T$, meaning *no* new term. So $\text{reward\_sum}(\tau+1)=S$ exactly.

Substituting $S$ back in:
$$\text{reward\_sum}(\tau+1) = \frac{\text{reward\_sum}(\tau) - R_{\tau+1}}{\gamma} + \begin{cases}\gamma^{n-1}R_{\tau+n+1} & \text{if }\tau+n+1\le T\\0&\text{otherwise}\end{cases}$$
Intuitively: $R_{\tau+1}$ (weight $\gamma^0$) is exactly the term the window is dropping, so it's subtracted before anything else; dividing by $\gamma$ then "ages" every surviving term by one more power, matching $\tau+1$'s frame; and the new term entering at the far end, $R_{\tau+n+1}$, is added at its correct weight $\gamma^{n-1}$ in that new frame -- *if* it exists (the window isn't already clipped by the episode's end).

**What $\text{reward\_sum}(\tau)$ actually is, relative to quantities from sections 1 and 9.** It is *not* $G_\tau$ (section 9's full return), nor is it quite $G_{\tau:\tau+n}$ (section 1's $n$-step return target) -- it's the reward-only piece shared by both:
$$\text{reward\_sum}(\tau) + \gamma^n\,V(S_{\tau+n}) = G_{\tau:\tau+n} \qquad\text{(bootstrapped target -- the estimate, what the code actually computes)}$$
$$\text{reward\_sum}(\tau) + \gamma^n\,G_{\tau+n} = G_\tau \qquad\text{(true full return -- section 9's identity, exact)}$$
Same $\text{reward\_sum}(\tau)$ both times; the only difference is whether "everything after $\tau+n$" is the *estimate* $V(S_{\tau+n})$ or the *true* continuation $G_{\tau+n}$. That's $n$-step TD in one picture: the real, observed reward for $n$ real steps, plus a current guess standing in for whatever the episode hasn't revealed yet. The two formulas -- and so $\text{reward\_sum}(\tau)$ itself -- coincide with $G_\tau$ and with $G_{\tau:\tau+n}$ at exactly one point: when $\tau+n\ge T$, there's no bootstrap to add (the pseudocode's own "if $\tau+n<T$" fails) *and* $G_{\tau+n}=G_T=0$, so all three collapse to the same thing.

**On the division actually being safe here, empirically:** checked across $\gamma$ from $1.0$ down to $0.05$, with episodes as long as 500+ real steps -- the discrepancy against the exact $O(n)$ recomputation stays at $\sim10^{-16}$-$10^{-17}$, floating-point noise, not real divergence. The reason it doesn't blow up despite the repeated division: this is a **fixed-size** window ($n$ terms), not an unbounded accumulation -- every division is immediately offset by dropping the term that just left and adding a correctly-rescaled new one, so the window's own magnitude stays bounded by the rewards themselves, rather than drifting toward $\gamma^{-\tau}$ over the whole episode. (Different from an eligibility trace accumulated over an *unbounded* horizon, where that blowup risk would be real.)

Structurally, this version keeps the same online $t$/$\tau$ interleaving as the original pseudocode-faithful `n_step_td_evaluate_episode` (unlike section 9's offline suffix-sum version) -- it's the same algorithm, just with an $O(1)$ sliding-window update to `reward_sum` in place of a fresh $O(n)$ recomputation at every $\tau$.

In [22]:
def n_step_td_evaluate_episode_sliding_window(values, n, alpha=0.1, gamma=1.0):
    """Same (t, tau) online structure as n_step_td_evaluate_episode, but
    the reward-sum part of G is maintained incrementally via the sliding-
    window recursion above (O(1) per update) instead of recomputed from
    scratch (O(n) per update). Works for any gamma in (0, 1] -- see the
    markdown above for the numerical-stability check."""
    state = START_STATE
    states = [state]
    rewards = [None]
    T = float("inf")
    t = 0
    reward_sum = None  # Sigma_{i=tau+1}^{min(tau+n,T)} gamma^{i-tau-1} R_i, for the current tau

    while True:
        if t < T:
            next_state, reward = step(state)
            states.append(next_state)
            rewards.append(reward)
            if next_state in TERMINAL_STATES:
                T = t + 1
            state = next_state

        tau = t - n + 1
        if tau >= 0:
            upper = min(tau + n, T)
            if reward_sum is None:
                reward_sum = 0.0
                for i in range(tau + 1, upper + 1):
                    reward_sum += gamma ** (i - tau - 1) * rewards[i]
            else:
                reward_sum = (reward_sum - rewards[tau]) / gamma   # R_tau leaves the window
                if tau + n <= T:
                    reward_sum += gamma ** (n - 1) * rewards[tau + n]  # R_{tau+n} enters it, if it exists
            G = reward_sum
            if tau + n < T:
                G += gamma ** n * values[states[tau + n]]
            s_tau = states[tau]
            values[s_tau] += alpha * (G - values[s_tau])

        if tau == T - 1:
            break
        t += 1

    return states, rewards

In [23]:
# Cross-check against both earlier versions: same seed, same values, for
# several n and several gamma (including small gamma, where the repeated
# division could in principle be risky -- see the numerical-stability note
# above) across many seeds. Just one episode per seed is enough -- this is
# checking that three implementations of the same algorithm agree with each
# other, not checking convergence, so there's nothing extra to gain from
# training longer (confirmed: the match is equally tight at 1 episode as at
# 30 -- see the chat discussion that led to this change).
mismatches = 0
worst_diff = 0.0
for gamma in [1.0, 0.99, 0.95, 0.9, 0.7, 0.5]:
    for n in [1, 2, 3, 5, 10, 25, 10_000]:
        for seed in range(50):
            np.random.seed(seed)
            v_pseudocode = np.zeros(21)
            n_step_td_evaluate_episode(v_pseudocode, n=n, alpha=0.1, gamma=gamma)

            np.random.seed(seed)
            v_suffix_sum = np.zeros(21)
            n_step_td_evaluate_episode_fast(v_suffix_sum, n=n, alpha=0.1, gamma=gamma)

            np.random.seed(seed)
            v_sliding = np.zeros(21)
            n_step_td_evaluate_episode_sliding_window(v_sliding, n=n, alpha=0.1, gamma=gamma)

            diff_a = np.max(np.abs(v_pseudocode - v_sliding))
            diff_b = np.max(np.abs(v_suffix_sum - v_sliding))
            worst_diff = max(worst_diff, diff_a, diff_b)
            if diff_a > 1e-9 or diff_b > 1e-9:
                mismatches += 1
                print(f"mismatch at gamma={gamma}, n={n}, seed={seed}, "
                      f"diff vs pseudocode={diff_a:.3e}, diff vs suffix-sum={diff_b:.3e}")

assert mismatches == 0
print(f"n_step_td_evaluate_episode_sliding_window matches both n_step_td_evaluate_episode "
      f"and n_step_td_evaluate_episode_fast across 6 gamma values x 7 n values x 50 seeds "
      f"(2,100 combinations), 1 episode per seed -- worst max abs diff: {worst_diff:.2e} "
      f"(floating-point noise).")

n_step_td_evaluate_episode_sliding_window matches both n_step_td_evaluate_episode and n_step_td_evaluate_episode_fast across 6 gamma values x 7 n values x 50 seeds (2,100 combinations) -- worst max abs diff: 6.66e-16 (floating-point noise).
